# KOA Onboarding from Bootstrap

Este notebook lê arquivos TXT de contratos (gerados a partir de seus PDFs) e gera modelos Prolog (um para cada contrato) com base no bootstrap.
A ideia é gerar os modelos Prolog dos contratos a partir de umaúnica ontologia fact-based, naive, gerada no passo anterior.


Entradas: contratos em .txt no diretório configurado.

Saídas: arquivos KOA_<contrato>.pl no diretório de persistência.

In [ ]:
# Monta o Google Drive no ambiente Colab para acessar os arquivos de contratos e salvar os .pl gerados
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# !pip uninstall google-generativeai -y
# !pip install google-genai

In [ ]:
# Importa bibliotecas padrão e do Gemini
from pathlib import Path
import os
import re
import importlib.util, types
import time

from google import genai

In [ ]:
# Recupera chave Gemini com secrets do Colab
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [ ]:
# Configura pastas para entradas e saídas
directory = '/content/drive/My Drive/KOA/contratos'
modelos_directory = '/content/drive/My Drive/KOA/onboarding/modelos'
persist_directory = '/content/drive/My Drive/KOA/onboarding/contratos_SEM_UFO'

In [ ]:
# Monta lista de ontologias que serão utilizadas no onboarding
bootstrap_file = {
    'KOA_ontology_bootstrap.pl'
}

In [ ]:
# Define função para leitura de arquivos
def read_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as file:
        return file.read()

In [ ]:
# Funções de apoio
def infer_contract_id(contract_text: str, stem: str) -> str:
    """
    Retorna contrato_ocs_XXX_YYYY.
    Estratégia:
      1) tenta achar padrão tipo '046/2024' no texto
      2) fallback: tenta achar no stem (nome do arquivo) algo parecido
    """
    # 1) do texto
    m = re.search(r"\b(\d{1,4})\s*/\s*(20\d{2})\b", contract_text)
    if m:
        num, year = m.group(1), m.group(2)
        return f"contrato_ocs_{num.zfill(3)}_{year}"

    # 2) fallback do nome do arquivo
    m = re.search(r"(\d{1,4})[_-]?(20\d{2})", stem)
    if m:
        num, year = m.group(1), m.group(2)
        return f"contrato_ocs_{num.zfill(3)}_{year}"

    # último fallback (não ideal)
    return f"contrato_ocs_{stem}"

def normalize_contract_id_in_prolog(pl_text: str, contract_id: str) -> str:
    # troca contract(<qualquer_coisa>).
    pl_text = re.sub(r"(?m)^\s*contract\(\s*[^)]+\s*\)\s*\.\s*$",
                     f"contract({contract_id}).",
                     pl_text)

    # troca o primeiro argumento dos predicados que carregam o contrato
    # contract_metadata(OLD, X, Y).
    pl_text = re.sub(r"(?m)^\s*contract_metadata\(\s*[^,]+\s*,",
                     f"contract_metadata({contract_id},",
                     pl_text)

    # contract_clause(OLD, ...)
    pl_text = re.sub(r"(?m)^\s*contract_clause\(\s*[^,]+\s*,",
                     f"contract_clause({contract_id},",
                     pl_text)

    # contract_clause_fact(OLD, ...)
    pl_text = re.sub(r"(?m)^\s*contract_clause_fact\(\s*[^,]+\s*,",
                     f"contract_clause_fact({contract_id},",
                     pl_text)

    # contract_signature(OLD, ...)
    pl_text = re.sub(r"(?m)^\s*contract_signature\(\s*[^,]+\s*,",
                     f"contract_signature({contract_id},",
                     pl_text)

    return pl_text

In [11]:
# Agente para geração do modelo - Funções de apoio
class ModelGenerationAgent:
    """
    Lê o texto do contrato e gera um modelo Prolog baseado em uma TBox (bootstrap) e um exemplo.
    Suporta geração multi-part para evitar o problema de truncamento da saída.
    """

    def __init__(self, file_path: str, bootstrap: str, example: str, contract_id: str,
                 model_name: str = "gemini-2.0-flash-lite",
                 sleep_seconds: int = 3,
                 max_parts: int = 6):
        self.file_path = file_path
        self.content = self._read_document()
        self.bootstrap = bootstrap
        self.example = example
        self.contract_id = contract_id
        self.prolog_model = None

        self.model_name = model_name
        self.sleep_seconds = sleep_seconds
        self.max_parts = max_parts

        # Configure Gemini
        self.client = genai.Client()

    def _read_document(self) -> str:
        try:
            with open(self.file_path, "r", encoding="utf-8") as f:
                return f.read()
        except Exception as e:
            print(f"Error reading '{self.file_path}': {e}")
            return ""

    @staticmethod
    def _strip_code_fences(text: str) -> str:
        # Remove ```prolog ... ``` or ``` ... ```
        text = re.sub(r"```(?:prolog)?\s*", "", text, flags=re.IGNORECASE)
        text = re.sub(r"\s*```", "", text)
        return text.strip()

    @staticmethod
    def _dedup_lines_keep_order(text: str) -> str:
        seen = set()
        out_lines = []
        for line in text.splitlines():
            key = line.strip()
            if not key:
                out_lines.append(line)
                continue
            if key in seen:
                continue
            seen.add(key)
            out_lines.append(line)
        return "\n".join(out_lines).strip() + "\n"

    def _call_gemini(self, prompt: str) -> str:
        print("Calling Gemini...")
        print(f"Prompt size: {len(prompt):,} chars")

        # Basic retry for transient rate limits
        for attempt in range(5):
            try:
                client = self.client
                resp = client.models.generate_content(
                    model='gemini-2.0-flash',
                    contents=prompt
                )
                print("Gemini responded")
                text = getattr(resp, "text", "") or ""
                print(f"Response size: {len(text):,} chars")
                return text
            except Exception as e:
                msg = str(e)
                if "429" in msg or "Resource exhausted" in msg:
                    wait = self.sleep_seconds * (attempt + 1)
                    print(f"⚠️ Rate limit. Sleeping {wait}s...")
                    time.sleep(wait)
                else:
                    raise
        return ""

    def _split_contract_into_clauses(self, text: str):
        """Return list of tuples (header_text, clause_text_including_header).

        Detect clause headers anywhere in the text, like:
          CLÁUSULA <ordinal words> – <TITLE>
        The ordinal part is kept flexible (any words in ALL CAPS with accents).
        """
        header_re = re.compile(
            r"(?i)(CLÁUSULA\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇÀÜ]+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇÀÜ]+)*\s*[–-]\s*.+?)"
            # r"(?i)(CLÁUSULA\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇÀÜ]+(?:\s+[A-ZÁÉÍÓÚÂÊÔÃÕÇÀÜ]+)*\s*[–-]\s*.+?)(?=\s+CLÁUSULA\s+|$)"
        )
        matches = list(header_re.finditer(text))
        if not matches:
            return "", []

        preamble = text[:matches[0].start()].strip()

        clauses = []
        for i, m in enumerate(matches):
            start = m.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
            header = m.group(1).strip()
            chunk = text[start:end].strip()
            clauses.append((header, chunk))

        return preamble, clauses

    def _chunk_clauses(self, clauses, max_chars: int = 12000):
        """Group consecutive clauses into chunks up to max_chars."""
        chunks = []
        cur = []
        cur_len = 0
        for header, body in clauses:
            piece = body.strip() + "\n"
            if cur and (cur_len + len(piece) > max_chars):
                chunks.append(cur)
                cur = []
                cur_len = 0
            cur.append((header, body))
            cur_len += len(piece)
        if cur:
            chunks.append(cur)
        return chunks

    def _build_chunk_prompt(self, clause_chunk, part_idx: int, total_parts: int) -> str:
        """Prompt for a single chunk of clauses (map step)."""
        headers = "\n".join([f"- {h}" for h, _ in clause_chunk])
        chunk_text = "\n\n".join([b for _, b in clause_chunk]).strip()
        is_preamble = clause_chunk and clause_chunk[0][0].startswith("PREÂMBULO")

        # Important: the example is only to show the format, NOT to limit generation
        return f"""You are a knowledge representation and Prolog expert.
        Based on the following ontology definition and the contract text, generate a formal Prolog model.
        The output must be ONLY Prolog code (no explanations, no markdown fences).
        Use only Portuguese identifiers/labels when applicable.

        IMPORTANT ABOUT THE EXAMPLE GIVEN:
        - The example below are ONLY to demonstrate the OUTPUT FORMAT and style.
        - They are NOT an exhaustive list of clauses, titles, fact types, or roles.
        - Do NOT limit extraction to the example clause numbers/titles/facts.
        - Always process ALL clauses provided in the input, from start to finish, and extract every relevant fact you can find in each clause.

        IMPORTANT CONTRACT ID RULE:
        - Use EXACTLY this contract id: {self.contract_id}
        - The first argument of ALL these predicates must be {self.contract_id}:
          contract/1, contract_metadata/3, contract_clause/4, contract_clause_fact/5, contract_signature/2.
        - Do not introduce any other contract identifier.

        CRITICAL OUTPUT RULES:
        - Output ONLY Prolog code.
        - Prefer one fact per line ending with a period.
        - Do NOT include comments unless they are already present in the example style.
        - Ensure the output is syntactically valid Prolog.
        - Schema enforcement (MANDATORY):
          * Use ONLY predicates listed in the Ontology Definition (predicate_signature/2).
          * For contract_metadata/3, Key MUST be one of allowed_metadata_key/1 in the Ontology Definition.
            - If the metadata key is not allowed, use: contract_metadata_raw(ContractId, KeyText, Value, Evidence).
          * Do NOT invent new predicate names beyond the allowed ones and the *_raw fallbacks.

        CLAUSE HEADER DETECTION (MANDATORY):
        - Treat the text that appears BEFORE the first detected clause header as the PREAMBLE.
          - From the PREAMBLE you MUST extract:
              a) contract_metadata/3 (use allowed keys),
              b) contract_metadata_raw/4 for any key not covered by allowed keys,
              c) contract_signature/2 if signatures/digital signature lines appear there.
          - Do NOT output contract_clause/4 or contract_clause_fact/5 for the PREAMBLE.
        - Treat EVERY occurrence of the ALL-CAPS token "CLÁUSULA" as the start of a clause header, EVEN if it appears mid-paragraph.
        - Accept ANY Portuguese ordinal written in full (e.g., PRIMEIRA..NONA, DÉCIMA, DÉCIMA PRIMEIRA..DÉCIMA NONA, VIGÉSIMA, VIGÉSIMA PRIMEIRA, VIGÉSIMA SEGUNDA, etc.).
        - Do NOT assume the contract ends at "DISPOSIÇÕES FINAIS": there may be clauses after it (e.g., "FORO").
        - For EACH detected clause header, you MUST output:
            1) contract_clause/4 with the full clause text (until right before the next header),
            2) all contract_clause_fact/5 extracted from that clause.
        - Clause names, ordinal and clause text MUST BE always in Portuguese.
        - If you run out of space, continue in the next part WITHOUT skipping any remaining clauses.

        Ontology Definition:
        ---
        {self.bootstrap}
        ---

        Output Example:
        ---
        {self.example}
        ---

        Text to analyze:
        ---
        {chunk_text}
        ---

        Prolog Model:
        """.strip()

    def _generate_model_multi_part(self) -> str:
        """Map-reduce style generation.

        1) Split the contract into clause blocks (chunking BEFORE calling the model)
        2) Call the model once per chunk (map)
        3) Merge + de-dup facts (reduce)
        """
        # Prefer using the already-cleaned content used in prompts
        base_text = (self.content or "").strip()

        max_parts = 20
        max_chars = 12000  # keep conservative to avoid model context issues

        preamble, clauses = self._split_contract_into_clauses(base_text)

        chunks = self._chunk_clauses(clauses, max_chars=max_chars)
        chunks = chunks[: self.max_parts]

        # insere “chunk 0” antes
        if preamble:
            preamble_chunk = [("PREÂMBULO – METADADOS", preamble)]
            chunks = [preamble_chunk] + chunks

        parts = []
        total = len(chunks)
        for idx, chunk in enumerate(chunks, start=1):
            if idx > 1:
                time.sleep(self.sleep_seconds)
            prompt = self._build_chunk_prompt(chunk, idx, total)
            pi = self._call_gemini(prompt)
            pi = self._strip_code_fences(pi)
            if len(pi.strip()) < 20:
                continue
            parts.append(pi)

        combined = "\n".join(parts).strip()
        combined = self._dedup_lines_keep_order(combined)
        return combined


    def generate_prolog_model(self) -> str:
        self.prolog_model = self._generate_model_multi_part()
        return self.prolog_model


In [12]:
if __name__ == "__main__":

    pl_dir = Path(directory)
    if not pl_dir.exists():
        raise FileNotFoundError(f"Directory not found: {pl_dir}")

    # Files used as bootstrap + example
    bootstrap_pl_path = Path(modelos_directory) / "KOA_ontology_bootstrap_v2.pl"
    origin_example_pl_path = Path(modelos_directory) / "KOA_195_2022_Brasoftware.pl"

    if not bootstrap_pl_path.exists():
        raise FileNotFoundError(f"Bootstrap TTL not found: {bootstrap_pl_path}")
    if not origin_example_pl_path.exists():
        raise FileNotFoundError(f"Origin example .pl not found: {origin_example_pl_path}")

    with open(bootstrap_pl_path, "r", encoding="utf-8") as f:
        ontology_bootstrap = f.read()

    with open(origin_example_pl_path, "r", encoding="utf-8") as f:
        model_example = f.read()

    # Normalize the excluded filename (case-insensitive)
    excluded_name = origin_example_pl_path.name.lower()

    # -----------------------------
    # Find all .pl files and process
    # -----------------------------
    pl_files = sorted(pl_dir.glob("Contrato*.txt"), key=lambda p: p.name.lower())

    if not pl_files:
        print(f"⚠️ No .txt contract files found in: {pl_dir}")

    for pl_path in pl_files:
        # Skip the origin file used as example for the bootstrap
        if pl_path.name.lower() == excluded_name:
            print(f"Skipping origin bootstrap file: {pl_path.name}")
            continue

        print("\n====================================================")
        print(f"Processing Prolog model: {pl_path.name}")

        if not pl_path.exists():
            print(f"⚠️  Skipping (file not found): {pl_path}")
            continue

        _doc_text = read_file(str(pl_path))

        stem = pl_path.stem.replace(" ", "_")
        contract_id = infer_contract_id(_doc_text, stem)
        print(f"Contract ID: {contract_id}")

        print("Generating base model ...")
        model_generator = ModelGenerationAgent(
            file_path=str(pl_path),
            bootstrap=ontology_bootstrap,
            example=model_example,
            contract_id=contract_id
        )
        base_model = model_generator.generate_prolog_model()
        # Anchor clause headers/bodies to the true contract text (prevents example/template drift)
        # truth_map = build_truth_map(extract_clauses(_doc_text))
        # if truth_map:
        #     base_model = repair_contract_clause_headers(base_model, truth_map)
        base_out = Path(persist_directory) / f"KOA_{stem}.pl"
        with open(base_out, "w", encoding="utf-8") as f:
            f.write(base_model)

print(f"Base ontology saved to '{base_out}'")



Processing Prolog model: Contrato OCS 008_2022 - Hitachi Vantara Storage.txt
Contract ID: contrato_ocs_008_2022
Generating base model ...
Calling Gemini...
Prompt size: 51,407 chars
Gemini responded
Response size: 1,740 chars
Calling Gemini...
Prompt size: 53,641 chars
Gemini responded
Response size: 6,845 chars
Calling Gemini...
Prompt size: 58,219 chars
Gemini responded
Response size: 10,767 chars
Calling Gemini...
Prompt size: 58,107 chars
Gemini responded
Response size: 12,008 chars
Calling Gemini...
Prompt size: 60,064 chars
Gemini responded
Response size: 12,544 chars
Calling Gemini...
Prompt size: 60,196 chars
Gemini responded
Response size: 11,797 chars
Calling Gemini...
Prompt size: 59,656 chars
Gemini responded
Response size: 12,734 chars

Processing Prolog model: Contrato OCS 011_2022 - Zoom (Storage Huawei).txt
Contract ID: contrato_ocs_0011_2022
Generating base model ...
Calling Gemini...
Prompt size: 51,361 chars
Gemini responded
Response size: 1,754 chars
Calling Gemini